Data extraction

In [ ]:
# // --- 1. Setup ---
# // Define your Area of Interest (AOI)
# var aoi = ee.Geometry.Rectangle([78.75, 15.78, 80.25, 17.22]);
# var startDate = '2017-01-01';
# var endDate = '2025-12-31'; // Set end date to get all 2024-2025 data
# Map.centerObject(aoi, 9);
# Map.addLayer(aoi, {color: 'black'}, 'Nagarjuna Sagar AOI', true);

# // Load GLDAS and select the groundwater proxy band
# var gldas = ee.ImageCollection('NASA/GLDAS/V021/NOAH/G025/T3H')
#               .filterDate(startDate, endDate)
#               .select('SoilMoi100_200cm_inst'); // <-- FIX 1: Added '_inst' suffix

# // Visualization Parameters
# var gwVis = {
#   min: 150,
#   max: 450,
#   palette: ['#FF0000', '#FFFFFF', '#0000FF'] // Red (dry) to Blue (wet)
# };

# // --- 2. Generate Yearly Seasonal Images ---
# var years = ee.List([2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]);

# // Create a collection of all MONSOON seasons
# var monsoonImages = ee.ImageCollection.fromImages(
#   years.map(function(y) {
#     var year = ee.Number(y);
#     var startDate = ee.Date.fromYMD(year, 6, 1); // June 1
#     var endDate = ee.Date.fromYMD(year, 9, 30); // Sept 30

#     var mean = gldas
#       .filterDate(startDate, endDate)
#       .mean()
#       .clip(aoi)
#       .set({
#         'year': year, // Standardized property name
#         'season': 'Monsoon',
#         'system:time_start': startDate.millis()
#       });

#     return mean;
#   })
# );

# // Create a collection of all DRY seasons
# var dryImages = ee.ImageCollection.fromImages(
#   years.map(function(y) {
#     var year = ee.Number(y);
#     var startDate = ee.Date.fromYMD(year, 10, 1); // Oct 1
#     var endDate = ee.Date.fromYMD(year.add(1), 5, 31); // May 31 (next year)

#     var mean = gldas
#       .filterDate(startDate, endDate)
#       .mean()
#       .clip(aoi)
#       .set({
#         'year': year, // <-- FIX 2: Standardized property to 'year'
#         'season': 'Dry',
#         'system:time_start': startDate.millis()
#       });

#     return mean;
#   })
# );

# // Merge the two collections into one
# var seasonalImages = monsoonImages.merge(dryImages);

# print('Yearly Seasonal Image Collection:', seasonalImages);

# // Add the most recent Monsoon and Dry season to the map as an example
# Map.addLayer(
#   seasonalImages.filterMetadata('year', 'equals', 2024)
#                 .filterMetadata('season', 'equals', 'Monsoon')
#                 .first(),
#   gwVis,
#   'Monsoon 2024'
# );
# Map.addLayer(
#   seasonalImages.filterMetadata('year', 'equals', 2024) // <-- FIX 3: Changed 'year_start' to 'year'
#                 .filterMetadata('season', 'equals', 'Dry')
#                 .first(),
#   gwVis,
#   'Dry 2024-2025'
# );


# // --- 3. Generate Monthly Time-Series Chart (RECOMMENDED) ---
# var months = ee.Date(endDate).difference(ee.Date(startDate), 'month').ceil();
# var monthList = ee.List.sequence(0, months.subtract(1));
# var start = ee.Date(startDate);

# var byMonth = ee.ImageCollection(
#   monthList.map(function(n) {
#     var monthStart = start.advance(n, 'month');
#     var monthEnd = monthStart.advance(1, 'month');
#     var monthlyMean = gldas
#                       .filterDate(monthStart, monthEnd)
#                       .mean()
#                       .set('system:time_start', monthStart.millis());
#     return monthlyMean;
#   })
# );

# var chart = ui.Chart.image.series({
#   imageCollection: byMonth,
#   region: aoi,
#   reducer: ee.Reducer.mean(),
#   scale: 27830, // Native resolution of GLDAS
#   xProperty: 'system:time_start'
# }).setOptions({
#   title: 'Monthly Shallow Groundwater Proxy (Soil Moisture 100-200cm)',
#   vAxis: {title: 'Soil Moisture (kg/m^2)'},
#   hAxis: {title: 'Date', format: 'MMM-yyyy'},
#   lineWidth: 1,
#   pointSize: 2,
#   series: { 0: {color: 'blue'} }
# });

# print(chart);


# // --- 4. (Optional) Export All Seasonal Images to Google Drive ---
# var imageList = seasonalImages.toList(seasonalImages.size());

# // Client-side loop to launch export tasks
# for (var i = 0; i < 16; i++) {
#   var image = ee.Image(imageList.get(i));

#   // <-- FIX 4: Simplified logic by using standardized properties -->
#   var year = ee.Number(image.get('year'));
#   var season = ee.String(image.get('season'));

#   // Create a server-side string for the description
#   var description = ee.String('Groundwater_')
#                       .cat(season)
#                       .cat('_')
#                       .cat(year.format('%d')); // This will now work

#   Export.image.toDrive({
#     image: image.toFloat(),
#     description: description.getInfo(), // Get the string value
#     folder: 'GEE_Seasonal_Data_Nagarjuna_S2_Second',
#     scale: 27830,
#     region: aoi,
#     crs: 'EPSG:4326',
#     fileFormat: 'GeoTiff'
#   });
# }

Structure

In [ ]:
!ls /content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second/Ground_water

In [ ]:
!ls /content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second/Rivers2

In [ ]:
!ls /content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second/Reservoirs2

Analysis

In [ ]:
# Install required libraries
!pip install geopandas rasterio

# Import libraries
import os
import glob
import re
import pandas as pd
import geopandas as gpd
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

In [ ]:
# --- Define Paths ---
gw_path = "/content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second/Ground_water"
rivers_path = "/content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second/Rivers2"
reservoirs_path = "/content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second/Reservoirs2"

# --- Find Files ---
# Find all groundwater .tif files
gw_files = glob.glob(os.path.join(gw_path, "Groundwater_*.tif"))

# Find all river .shp files
river_files = glob.glob(os.path.join(rivers_path, "rivers_*.shp"))

# Find all reservoir .shp files
reservoir_files = glob.glob(os.path.join(reservoirs_path, "reservoirs_*.shp"))

# Sort files to ensure correct time order
gw_files.sort()
river_files.sort()
reservoir_files.sort()

print(f"Found {len(gw_files)} groundwater files.")
print(f"Found {len(river_files)} river shapefiles.")
print(f"Found {len(reservoir_files)} reservoir shapefiles.")
print("\nSample GW file:", gw_files[0])
print("Sample River file:", river_files[0])

In [ ]:
gw_data = []

# Regex to parse year and season from filename
gw_pattern = re.compile(r"Groundwater_(\w+)_(\d{4})\.tif")

for f in gw_files:
    # Parse filename
    match = gw_pattern.search(os.path.basename(f))
    if not match:
        continue

    season, year = match.groups()

    # Read the raster file
    with rasterio.open(f) as src:
        array = src.read(1)
        nodata = src.nodata

        # Mask out nodata values (if any, otherwise mask 0s)
        if nodata is not None:
            array_masked = array[array != nodata]
        else:
            array_masked = array[array > 0] # GLDAS data should be > 0

        # Calculate the mean value for the entire raster
        if array_masked.size > 0:
            mean_val = array_masked.mean()
        else:
            mean_val = np.nan

    gw_data.append({
        "year": int(year),
        "season_raw": season,
        "gw_mean_proxy": mean_val
    })

# Convert to DataFrame
gw_df = pd.DataFrame(gw_data)

print("--- Groundwater DataFrame ---")
print(gw_df.head())

In [ ]:
def process_vectors(file_list, file_prefix):
    """
    Processes a list of shapefiles to calculate their total area.
    - file_list: List of paths to .shp files.
    - file_prefix: The prefix to remove (e.g., "rivers_" or "reservoirs_")
    """
    vector_data = []

    # Regex to parse year and season
    # e.g., rivers_2018_01_Winter_...
    pattern = re.compile(rf"{file_prefix}_(\d{{4}})_(\d{{2}})_(\w+)_")

    for f in file_list:
        match = pattern.search(os.path.basename(f))
        if not match:
            continue

        year, season_idx, season = match.groups()

        # Read shapefile
        gdf = gpd.read_file(f)

        # --- CRITICAL: Calculate Area ---
        # 1. Reproject to a projected CRS (UTM Zone 44N for Nagarjuna Sagar)
        #    to get area in meters instead of degrees.
        gdf_projected = gdf.to_crs("EPSG:32644")

        # 2. Calculate area in square meters, sum it, and convert to sq km
        total_area_sqkm = gdf_projected.geometry.area.sum() / 1_000_000

        vector_data.append({
            "year": int(year),
            "season_raw": season,
            "area_sqkm": total_area_sqkm
        })

    return pd.DataFrame(vector_data)

# --- Process both datasets ---
rivers_df = process_vectors(river_files, "rivers")
reservoirs_df = process_vectors(reservoir_files, "reservoirs")

print("\n--- Rivers DataFrame ---")
print(rivers_df.head())
print("\n--- Reservoirs DataFrame ---")
print(reservoirs_df.head())

In [ ]:
# --- 1. Clean Groundwater Data ---
# Rename 'Dry' to 'DrySeason' for clarity
gw_df['season'] = gw_df['season_raw'].map({
    'Monsoon': 'Monsoon',
    'Dry': 'DrySeason'
})
# For GW 'Dry_2017' (Oct'17-May'18), we map it to year 2018 to match river data
gw_df['year_match'] = np.where(gw_df['season'] == 'DrySeason', gw_df['year'] + 1, gw_df['year'])
gw_df_clean = gw_df.drop(columns=['season_raw', 'year'])


# --- 2. Clean River/Reservoir Data ---
def clean_surface_water_df(df, col_name):
    # --- FIX: Extract base season name ---
    # Extracts 'Monsoon', 'Winter', or 'Summer' from strings like 'Monsoon_NDWI_MNDWI'
    df['season_base'] = df['season_raw'].str.extract(r'(Monsoon|Winter|Summer)')

    # Now, map the extracted base names
    df['season'] = df['season_base'].map({
        'Monsoon': 'Monsoon',
        'Winter': 'DrySeason',
        'Summer': 'DrySeason'
    })

    # Average 'Winter' and 'Summer' to get one 'DrySeason' value per year
    df_agg = df.groupby(['year', 'season'])['area_sqkm'].mean().reset_index()

    # Rename area column for merging
    df_agg = df_agg.rename(columns={'area_sqkm': col_name})
    return df_agg

rivers_df_clean = clean_surface_water_df(rivers_df, 'river_area_sqkm')
reservoirs_df_clean = clean_surface_water_df(reservoirs_df, 'reservoir_area_sqkm')

print("\n--- Debug: Post-fix rivers_df_clean (head) ---")
print(rivers_df_clean.head())


# --- 3. Merge All DataFrames (back to 'inner') ---
# Now that the keys match, we can use 'inner' to keep only complete rows
df_merged = pd.merge(
    gw_df_clean,
    rivers_df_clean,
    left_on=['year_match', 'season'],
    right_on=['year', 'season'],
    how='inner' # <-- Back to 'inner'
)

# Merge the result with Reservoirs
df_final = pd.merge(
    df_merged,
    reservoirs_df_clean,
    on=['year', 'season'],
    how='inner'
)

# Clean up final dataframe
df_final = df_final[['year', 'season', 'gw_mean_proxy', 'river_area_sqkm', 'reservoir_area_sqkm']].drop_duplicates()

print("\n--- Final Merged DataFrame ---")
print(df_final.head())

# Optional: Check if anything was dropped
if df_final.empty:
    print("\nWARNING: Merge still failed. Check year ranges.")
else:
    print(f"\nSuccessfully merged {len(df_final)} rows.")

In [ ]:
# Create a datetime column for plotting
# We'll set Monsoon to August and DrySeason to March for visual spacing
date_map = {
    'Monsoon': '-08-01',
    'DrySeason': '-03-01'
}
df_final['date'] = pd.to_datetime(df_final['year'].astype(str) + df_final['season'].map(date_map))
df_final = df_final.sort_values('date')

# --- Create the plot ---
fig, ax1 = plt.subplots(figsize=(15, 7))

# Plot Groundwater on primary (left) axis
ax1.plot(df_final['date'], df_final['gw_mean_proxy'], 'o-', color='blue', label='Groundwater Proxy (kg/m²)')
ax1.set_xlabel('Date')
ax1.set_ylabel('Mean Groundwater Proxy (Soil Moisture)', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

# Create a secondary (right) axis for Area
ax2 = ax1.twinx()
ax2.plot(df_final['date'], df_final['reservoir_area_sqkm'], 's--', color='green', label='Reservoir Area (sqkm)')
ax2.plot(df_final['date'], df_final['river_area_sqkm'], '^-', color='red', label='River Area (sqkm)', alpha=0.7)
ax2.set_ylabel('Water Surface Area (sqkm)', color='green')
ax2.tick_params(axis='y', labelcolor='green')

# Add title and legend
plt.title('Groundwater vs. Surface Water Area (2018-2024)', fontsize=16)
fig.legend(loc="upper right", bbox_to_anchor=(1,1), bbox_transform=ax1.transAxes)
plt.grid(True)
plt.show()

In [ ]:
# Separate the data by season
df_monsoon = df_final[df_final['season'] == 'Monsoon']
df_dry = df_final[df_final['season'] == 'DrySeason']

# --- Create Subplots ---
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('Correlation Analysis (Groundwater vs. Surface Water)', fontsize=16)

# 1. Monsoon: GW vs. Reservoir
ax1.scatter(df_monsoon['gw_mean_proxy'], df_monsoon['reservoir_area_sqkm'], color='blue')
ax1.set_title('Monsoon: GW vs. Reservoir Area')
ax1.set_xlabel('Groundwater Proxy')
ax1.set_ylabel('Reservoir Area (sqkm)')
ax1.grid(True)

# 2. Monsoon: GW vs. River
ax2.scatter(df_monsoon['gw_mean_proxy'], df_monsoon['river_area_sqkm'], color='blue')
ax2.set_title('Monsoon: GW vs. River Area')
ax2.set_xlabel('Groundwater Proxy')
ax2.set_ylabel('River Area (sqkm)')
ax2.grid(True)

# 3. Dry Season: GW vs. Reservoir
ax3.scatter(df_dry['gw_mean_proxy'], df_dry['reservoir_area_sqkm'], color='red')
ax3.set_title('Dry Season: GW vs. Reservoir Area')
ax3.set_xlabel('Groundwater Proxy')
ax3.set_ylabel('Reservoir Area (sqkm)')
ax3.grid(True)

# 4. Dry Season: GW vs. River
ax4.scatter(df_dry['gw_mean_proxy'], df_dry['river_area_sqkm'], color='red')
ax4.set_title('Dry Season: GW vs. River Area')
ax4.set_xlabel('Groundwater Proxy')
ax4.set_ylabel('River Area (sqkm)')
ax4.grid(True)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
def calculate_correlation(df, col1, col2):
    # Drop any NaN values to avoid errors
    df_clean = df[[col1, col2]].dropna()

    if len(df_clean) < 3: # Need at least 3 data points
        return (np.nan, np.nan)

    r, p = pearsonr(df_clean[col1], df_clean[col2])
    return r, p

# --- Monsoon Correlations ---
r_gw_res_monsoon, p_gw_res_monsoon = calculate_correlation(df_monsoon, 'gw_mean_proxy', 'reservoir_area_sqkm')
r_gw_riv_monsoon, p_gw_riv_monsoon = calculate_correlation(df_monsoon, 'gw_mean_proxy', 'river_area_sqkm')

# --- Dry Season Correlations ---
r_gw_res_dry, p_gw_res_dry = calculate_correlation(df_dry, 'gw_mean_proxy', 'reservoir_area_sqkm')
r_gw_riv_dry, p_gw_riv_dry = calculate_correlation(df_dry, 'gw_mean_proxy', 'river_area_sqkm')

# --- Print Results ---
print("--- Correlation Results (r-value, p-value) ---")
print(f"Monsoon GW vs. Reservoir:\t r={r_gw_res_monsoon:.3f}, p={p_gw_res_monsoon:.3f}")
print(f"Monsoon GW vs. River:\t\t r={r_gw_riv_monsoon:.3f}, p={p_gw_riv_monsoon:.3f}")
print(f"Dry Season GW vs. Reservoir:\t r={r_gw_res_dry:.3f}, p={p_gw_res_dry:.3f}")
print(f"Dry Season GW vs. River:\t\t r={r_gw_riv_dry:.3f}, p={p_gw_riv_dry:.3f}")

# --- Lag Analysis: Does Monsoon GW affect the *next* Dry Season's River? ---
df_lag = df_final.set_index('date').sort_index()
# Shift GW data by 1 period (Monsoon -> DrySeason)
df_lag['gw_lag_1_season'] = df_lag['gw_mean_proxy'].shift(1)

# Correlate lagged GW with dry season river area
df_lag_dry = df_lag[df_lag['season'] == 'DrySeason']
r_lag, p_lag = calculate_correlation(df_lag_dry, 'gw_lag_1_season', 'river_area_sqkm')
print(f"\nLagged (Monsoon GW vs. *Next* Dry Season River):\t r={r_lag:.3f}, p={p_lag:.3f}")

In [ ]:
# --- 1. Install Seaborn for advanced plots ---
!pip install seaborn

import seaborn as sns
from scipy.stats import pearsonr

# --- 2. Create the Lagged DataFrame ---
# This is the key to this "advanced" plot.
# We are creating a new table to test our hypothesis.

# Get all Monsoon data (the "deposit")
df_monsoon = df_final[df_final['season'] == 'Monsoon'][[
    'year', 'gw_mean_proxy'
]].rename(columns={'gw_mean_proxy': 'gw_monsoon_recharge'})

# Get all Dry Season data (the "withdrawal")
df_dry = df_final[df_final['season'] == 'DrySeason'][[
    'year', 'river_area_sqkm', 'reservoir_area_sqkm'
]]

# Shift the "Dry Season Year" to match the *previous* monsoon
# e.g., Dry Season 2018 will match Monsoon 2017
df_dry['year_of_recharge'] = df_dry['year'] - 1

# Merge on the monsoon year
df_lagged = pd.merge(
    df_monsoon,
    df_dry,
    left_on='year',
    right_on='year_of_recharge'
)

print("--- Lagged DataFrame for Analysis ---")
print(df_lagged.head())


# --- 3. Create the Plot ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle("Groundwater as a Hydrological 'Bank': The Lagged Effect", fontsize=16, y=1.03)

# --- Plot 1: GW Recharge vs. *Next* Dry Season River Area ---
# 'regplot' will automatically add a linear regression line and confidence interval
sns.regplot(
    data=df_lagged,
    x='gw_monsoon_recharge',
    y='river_area_sqkm',
    ax=ax1,
    color='blue',
    ci=95 # 95% confidence interval
)
ax1.set_title('Monsoon "Recharge" vs. Following Dry Season River Area', fontsize=12)
ax1.set_xlabel('Mean Monsoon Groundwater Proxy (kg/m²)')
ax1.set_ylabel('Mean Dry Season River Area (sqkm)')
ax1.grid(True)

# Add correlation (r-value) to the plot
r_river, p_river = pearsonr(df_lagged['gw_monsoon_recharge'], df_lagged['river_area_sqkm'])
ax1.text(0.05, 0.95, f'r = {r_river:.3f}\np = {p_river:.3f}',
         transform=ax1.transAxes, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))


# --- Plot 2: GW Recharge vs. *Next* Dry Season Reservoir Area ---
sns.regplot(
    data=df_lagged,
    x='gw_monsoon_recharge',
    y='reservoir_area_sqkm',
    ax=ax2,
    color='green',
    ci=95
)
ax2.set_title('Monsoon "Recharge" vs. Following Dry Season Reservoir Area', fontsize=12)
ax2.set_xlabel('Mean Monsoon Groundwater Proxy (kg/m²)')
ax2.set_ylabel('Mean Dry Season Reservoir Area (sqkm)')
ax2.grid(True)

# Add correlation (r-value) to the plot
r_res, p_res = pearsonr(df_lagged['gw_monsoon_recharge'], df_lagged['reservoir_area_sqkm'])
ax2.text(0.05, 0.95, f'r = {r_res:.3f}\np = {p_res:.3f}',
         transform=ax2.transAxes, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()